In [1]:
# ===========================================================
# F1M1 — Módulo 1: EDA Leonali
# Notebook 09 — Análisis ABC y Pareto
# ===========================================================

import pandas as pd
import plotly.express as px
import sys
sys.path.insert(0, '..')
from src.kpis import pareto_80_20

df = pd.read_parquet('../data/processed/Dataset1_anonimizado.parquet')
df.shape

(581306, 15)

In [2]:
pareto = pareto_80_20(df)
pareto.head(10)

,Producto,Costo_Venta,acumulado,es_80
92,Espinaca Baby 250gr,5.974864e+08,18.874186,True
87,Espinaca 200gr,1.624579e+08,24.006120,True
76,Ens Oriental 300gr,1.591741e+08,29.034322,True
137,Mks Espinaca Baby 250gr,1.458973e+08,33.643119,True
193,Zanahoria Baby 150gr,1.258905e+08,37.619915,True
211,Zanahoria baby 340gr,1.251580e+08,41.573572,True
136,Mks Espinaca 280gr,1.051522e+08,44.895257,True
67,Ens Basica Mixta 250gr,1.015307e+08,48.102544,True
159,SF 5 200gr,8.383418e+07,50.750808,True
79,Ens Ranch 226gr,7.525532e+07,53.128072,True


In [3]:
import numpy as np

condiciones = [
    pareto['acumulado'] <= 80,
    pareto['acumulado'] <= 95,
    pareto['acumulado'] > 95
]
valores = ['A', 'B', 'C']

pareto['clase'] = np.select(condiciones, valores)

In [4]:
pareto['clase'].value_counts()

clase
C    146
B     39
A     29
Name: count, dtype: int64

In [5]:
pareto.groupby('clase')['Costo_Venta'].sum()


clase
A    2.526888e+09
B    4.765498e+08
C    1.621893e+08
Name: Costo_Venta, dtype: float64

In [6]:
resumen = pareto.groupby('clase').agg(
    productos=('Producto', 'count'),
    venta=('Costo_Venta', 'sum')
)
resumen['pct_catalogo'] = resumen['productos'] / resumen['productos'].sum() * 100
resumen['pct_venta'] = resumen['venta'] / resumen['venta'].sum() * 100
resumen

,productos,venta,pct_catalogo,pct_venta
clase,,,,
A,29,2.526888e+09,13.551402,79.822668
B,39,4.765498e+08,18.224299,15.053882
C,146,1.621893e+08,68.224299,5.123450


In [10]:
fig = px.line(
    pareto,
    x=range(1, len(pareto)+1),
    y='acumulado',
    title='Curva de Pareto - Leonali (214 productos)'
)
fig.add_hline(y=80, line_dash="dash", line_color="red", annotation_text="80% (Clase A)")
fig.add_hline(y=95, line_dash="dash", line_color="orange", annotation_text="95% (Clase B)")
fig.show()
fig.write_image('../reports/09_curva_pareto.png', width=1200, height=500)

In [11]:
fig = px.bar(
    resumen.reset_index(),
    x='clase',
    y='pct_venta',
    color='clase',
    title='Distribución ABC - % de Venta por Clase',
    text='pct_venta'
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.show()
fig.write_image('../reports/09_abc_distribucion.png', width=1200, height=500)

In [9]:
productos_a = pareto[pareto['clase'] == 'A'][['Producto', 'Costo_Venta', 'acumulado']]
productos_a

,Producto,Costo_Venta,acumulado
92,Espinaca Baby 250gr,5.974864e+08,18.874186
87,Espinaca 200gr,1.624579e+08,24.006120
76,Ens Oriental 300gr,1.591741e+08,29.034322
137,Mks Espinaca Baby 250gr,1.458973e+08,33.643119
193,Zanahoria Baby 150gr,1.258905e+08,37.619915
211,Zanahoria baby 340gr,1.251580e+08,41.573572
136,Mks Espinaca 280gr,1.051522e+08,44.895257
67,Ens Basica Mixta 250gr,1.015307e+08,48.102544
159,SF 5 200gr,8.383418e+07,50.750808
79,Ens Ranch 226gr,7.525532e+07,53.128072
